***Bronze Layer***

**Step 1 - Define Schema as String**

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("quantity", StringType(), True),
    StructField("unit_price", StringType(), True),
    StructField("discount", StringType(), True),
    StructField("total_amount", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("store_location", StringType(), True),
    StructField("order_status", StringType(), True)
])

customer_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("date_of_birth", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("registration_date", StringType(), True),
    StructField("loyalty_points", StringType(), True),
    StructField("status", StringType(), True)
])

**Step 2 - Read raw data as csv**

**Order Table**

In [0]:
orders_bronze_df = spark.read.format('csv').option("header", "true").schema(orders_schema).load("/Volumes/retail_project/raw_data/raw_file/retail_orders_messy.csv")
orders_bronze_df.display()

order_id,order_date,customer_id,product_id,product_name,category,quantity,unit_price,discount,total_amount,payment_method,store_location,order_status
O00001,13-02-2025,C0019,P210,Air Conditioner,Electronic,1,9314,0.05,null,Cash,Pune,Completed
O00002,22-01-2025,C0349,P207,Microwave Oven,Fashon,-1,abc,0.05,null,Credit Card,Chennai,Returned
O00003,2025-01-06,C0571,null,Refrigerator,electronics,-1,abc,0.05,null,Credit Card,Pune,Cancelled
O00004,2025-01-13,C0426,P201,iPhone 15,Fashon,3,61749,0.05,null,null,Pune,Completed
O00005,2025/02/24,C0640,P201,iPhone 15,electronics,3,abc,10%,null,Debit Card,Hyderabad,Returned
O00006,2025-02-16,C0157,P205,Levi Jeans,Fashion,3,55005,10%,null,Credit Card,Delhi,Cancelled
O00007,2025-01-15,C0336,P203,Sony Headphones,electronics,1,19012,0.1,null,Credit Card,Bangalore,completed
O00008,2025-03-12,C0597,P202,Samsung TV,Fashon,2,abc,0.1,null,null,Hyderabad,Cancelled
O00009,2025/03/26,C0103,P207,Microwave Oven,Electronics,1,abc,0,null,Credit Card,,Completed
O00010,2025-03-11,C0110,P208,Refrigerator,Fashon,1,abc,5%,null,Debit Card,Mumbai,Cancelled


**Customer Table**

In [0]:
customers_bronze_df = spark.read.format("csv") \
    .option("header", "true") \
    .schema(customer_schema) \
    .load("/Volumes/retail_project/raw_data/raw_file/retail_customers_messy.csv")
customers_bronze_df.display()    

customer_id,customer_name,email,phone,gender,date_of_birth,city,state,registration_date,loyalty_points,status
C0001,Neha Sharma,nehasharma@email.com,not_available,Female,04-03-1991,Delhi,Delhi,06-04-2023,1608,Active
C0002,Karan Gupta,karangupta@email.com,not_available,Male,06/08/1985,Jaipur,Rajasthan,12-10-2023,ten,Active
C0003,Sneha Verma,snehaverma@email.com,14097807,Male,1987/01/30,Jaipur,Maharashtra,2024/07/19,ten,Inactive
C0004,Anjali Kapoor,anjalikapoor@email.com,not_available,Male,1997-12-13,Mumbai,Uttar Pradesh,12/10/2023,null,Inactive
C0005,Sneha Singh,snehasingh@email.com,24678832,Male,1993/04/21,Bangalore,West Bengal,2024/01/24,null,Inactive
C0006,Karan Mehta,karanmehta@email.com,not_available,Female,2000-05-10,Chennai,West Bengal,10/02/2024,569,Active
C0007,Rohan Singh,rohansingh@email.com,24942594,Female,1990/07/14,Mumbai,Delhi,23-04-2023,null,Inactive
C0008,Amit Rao,amitrao@email.com,28499601,Male,18/05/1998,Kolkata,Telangana,2024/10/18,ten,Active
C0009,Rohan Rao,rohanrao@email.com,12933722,Male,1988/07/19,Bangalore,Rajasthan,2023-06-15,1125,Inactive
C0010,Sneha Iyer,snehaiyer@email.com,34516264,Male,1985/06/09,Lucknow,Maharashtra,29-06-2024,ten,Active


**Step 3 - Add Ingestion metadata**

In [0]:
from pyspark.sql.functions import current_timestamp


orders_bronze_df = orders_bronze_df.withColumn('ingest_timestamp', current_timestamp())
customers_bronze_df = customers_bronze_df.withColumn('ingest_timestamp',current_timestamp())

**Step -4 Save AS Delta Table in Bronze**

In [0]:
orders_bronze_df.write.format("delta").mode("overwrite").saveAsTable('retail_project.bronze.bronze_orders')
customers_bronze_df.write.format("delta").mode("overwrite").saveAsTable('retail_project.bronze.bronze_customers')